In [66]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap

df_obras = pd.read_csv('../../db/clean/obras-nao-publicitarias.csv')
df_fsa = pd.read_csv('../../db/cluster/obras_investimento_fsa.csv', sep=";")
df_coproducoes = pd.read_csv('../../db/cluster/coproducoes_brasileiras.csv', sep=";")
df_fomento_ind = pd.read_csv('../../db/cluster/obras_fomento_indireto.csv', sep=";")
df_lanc_comerciais = pd.read_csv('../../db/cluster/lancamentos_comerciais.csv', sep=";") # Remover?
df_dist = pd.read_csv('../../db/clean/bilheteria-distribuidoras.csv', sep=",")
df_exib = pd.read_csv('../../db/clean/bilheteria-exibidoras.csv', sep=",")
df_fsa.columns = df_fsa.columns.str.lower()
df_coproducoes.columns = df_coproducoes.columns.str.lower()
df_fomento_ind.columns = df_fomento_ind.columns.str.lower()
df_lanc_comerciais.columns = df_lanc_comerciais.columns.str.lower()

In [67]:
df_lanc_comerciais

,data_lancamento_obra,titulo_original,cpb_roe,tipo_obra,pais_obra,publico_total,renda_total,razao_social_distribuidora,registro_distribuidora,cnpj_distribuidora
0,26/03/2026,MORTE E VIDA MADALENA,B2400372900000,FICÇÃO,BRASIL,374,"R$ 4.973,27",EMBAUBA FILMES LTDA,23638.0,15.144.532/0001-70
1,06/11/2025,UMA NOVA HISTÓRIA,B2500383600000,FICÇÃO,BRASIL,1180,"R$ 22.046,80",HEAVEN CONTENT LTDA,62174.0,60.240.539/0001-40
2,30/10/2025,A MEMÓRIA DO CHEIRO DAS COISAS,B2500349200000,FICÇÃO,PORTUGAL,13,"R$ 152,51",TREZE DE MAIO DISTRIBUIDORA E PRODUTORA LTDA,52561.0,40.379.968/0001-95
3,30/10/2025,DREAMS,E2500328600000,FICÇÃO,ESTADOS UNIDOS,1799,"R$ 47.248,95",WMIX DISTRIBUIDORA LTDA.,1935.0,03.918.609/0001-32
4,30/10/2025,ENTERRE SEUS MORTOS,B2500154400000,FICÇÃO,BRASIL,1301,"R$ 26.009,28",O2 PRODUÇÕES ARTÍSTICAS E CINEMATOGRÁFICAS LTDA.,63.0,67.431.718/0001-03
...,...,...,...,...,...,...,...,...,...,...
6823,04/01/2009,THE STRANGERS,E1600633000000,FICÇÃO,ESTADOS UNIDOS,291713,"R$ 2.350.017,91",SM DISTRIBUIDORA DE FILMES LTDA,11922.0,08.257.054/0001-49
6824,02/01/2009,SE EU FOSSE VOCÊ 2,B0800958700000,FICÇÃO,BRASIL,29830,"R$ 37.557,00",CANNES PRODUÇÕES LTDA,2250.0,72.672.017/0001-04
6825,02/01/2009,SE EU FOSSE VOCÊ 2,B0800958700000,FICÇÃO,BRASIL,6113251,"R$ 50.545.885,00",FOX FILM DO BRASIL LTDA,256.0,33.110.420/0001-80
6826,01/01/2009,BOLT,E1500346000000,ANIMAÇÃO,ESTADOS UNIDOS,1162616,"R$ 9.590.392,25",COLUMBIA TRISTAR FILMES DO BRASIL LTDA,84.0,00.979.601/0001-98


In [68]:
# Remove os filmes estrangeiros de `df_lanc_comerciais`
df_lanc_comerciais = df_lanc_comerciais[~df_lanc_comerciais['cpb_roe'].str.startswith('E', na=False)]

# Renomeia a coluna `cpb_roe` apenas para `cpb`, já que não há mais registros estrangeiros
df_lanc_comerciais.rename(columns={'cpb_roe': 'cpb'}, inplace=True)



/tmp/ipykernel_2319744/4084595463.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lanc_comerciais.rename(columns={'cpb_roe': 'cpb'}, inplace=True)


In [69]:
# Trata o campo `renda_total`
df_lanc_comerciais = df_lanc_comerciais.copy()
df_lanc_comerciais['renda_total'] = df_lanc_comerciais['renda_total'] \
.str.replace('R$ ', '', regex=False) \
.str.replace('.', '', regex=False) \
.str.replace(',', '.', regex=False) \
.astype(float)

In [70]:
# Remove os filmes estrangeiros de `df_dist`
df_dist = df_dist[~df_dist['cpb_roe'].str.startswith('E', na=False)]

# Renomeia a coluna `cpb_roe` apenas para `cpb`, já que não há mais registros estrangeiros
df_dist.rename(columns={'cpb_roe': 'cpb'}, inplace=True)

In [71]:
# Remove os filmes estrangeiros de `df_exib`
df_exib = df_exib[~df_exib['cpb_roe'].str.startswith('E', na=False)]

# Renomeia a coluna `cpb_roe` apenas para `cpb`, já que não há mais registros estrangeiros
df_exib.rename(columns={'cpb_roe': 'cpb'}, inplace=True)

# Renomeia coluna `sessao`
df_exib.rename(columns={'sessao': 'data_exibicao'}, inplace=True)

In [72]:
# Remove registros do tipo "não classificado" do dataframe de obras
df_obras = df_obras[df_obras['tipo_obra'] != 'NÃO CLASSIFICADA']

In [73]:
df_cluster = pd.DataFrame()
df_cluster = df_obras[['cpb']].copy()
df_cluster.set_index('cpb', inplace=True)
df_cluster['investimento_fsa'] = pd.NA
df_cluster['coproducao'] = pd.NA
df_cluster['fomento_indireto'] = pd.NA
df_cluster['publico_total'] = pd.NA
df_cluster['qtd_sessoes'] = pd.NA
df_cluster['dias_cartaz'] = pd.NA
df_cluster['renda_total'] = pd.NA

In [74]:
# Adiciona valores na coluna `investimento_fsa`
fsa_cpbs = set(df_fsa['cpb'].dropna())
df_cluster['investimento_fsa'] = df_cluster.index.isin(fsa_cpbs)

In [75]:
# Adiciona valores na coluna `coproducao`
coproducao_cpbs = set(df_coproducoes['cpb'].dropna())
df_cluster['coproducao'] = df_cluster.index.isin(coproducao_cpbs)

In [76]:
# Adiciona valores na coluna `fomento_indireto`
fomento_ind_cpbs = set(df_fomento_ind['cpb'].dropna())
df_cluster['fomento_indireto'] = df_cluster.index.isin(fomento_ind_cpbs)

In [77]:
# Adiciona colunas para `cat_duracao`
map_cats = df_obras.set_index('cpb')['cat_duracao']

df_cluster['cat_duracao'] = df_cluster.index.map(map_cats)

df_cluster = pd.get_dummies(df_cluster, columns=['cat_duracao'])
hot_encode_nomes = {
    'cat_duracao_CURTA METRAGEM': 'curta_metragem',
    'cat_duracao_MÉDIA METRAGEM': 'media_metragem',
    'cat_duracao_LONGA METRAGEM': 'longa_metragem'
}
df_cluster = df_cluster.rename(columns=hot_encode_nomes)

In [78]:
df_obras['tipo_obra'].value_counts()

tipo_obra
VÍDEOMUSICAL    13106
DOCUMENTÁRIO     8991
FICÇÃO           8272
ANIMAÇÃO         1654
VARIEDADES        541
Name: count, dtype: int64

In [79]:
# Adiciona colunas para `tipo_obra`
map_tipo = df_obras.set_index('cpb')['tipo_obra']

df_cluster['tipo_obra'] = df_cluster.index.map(map_tipo)

df_cluster = pd.get_dummies(df_cluster, columns=['tipo_obra'])
hot_encode_nomes = {
    'tipo_obra_VÍDEOMUSICAL': 'videomusical',
    'tipo_obra_DOCUMENTÁRIO': 'documentario',
    'tipo_obra_FICÇÃO': 'ficcao',
    'tipo_obra_ANIMAÇÃO': 'animacao',
    'tipo_obra_VARIEDADES': 'variedades',
}
df_cluster = df_cluster.rename(columns=hot_encode_nomes)

In [80]:
# Adiciona a coluna `publico_total`
map_publico_comercial = df_lanc_comerciais.groupby('cpb')['publico_total'].sum()
map_publico_dist = df_dist.groupby('cpb')['publico'].sum()
map_publico_exib = df_exib.groupby('cpb')['publico'].sum()

s_comercial = map_publico_comercial.reindex(df_cluster.index)
s_dist = map_publico_dist.reindex(df_cluster.index)
s_exib = map_publico_exib.reindex(df_cluster.index)

df_cluster['publico_total'] = s_comercial.combine_first(s_dist).combine_first(s_exib)

In [81]:
# Adiciona a coluna `qtd_sessoes`
map_qtd_dist = df_dist.groupby('cpb').size()
map_qtd_exib = df_exib.groupby('cpb').size()

s_qtd_dist = map_qtd_dist.reindex(df_cluster.index)
s_qtd_exib = map_qtd_exib.reindex(df_cluster.index)

df_cluster['qtd_sessoes'] = s_qtd_dist.combine_first(s_qtd_exib)

In [82]:
# Adiciona a coluna `dias_cartaz`
df_dist = df_dist.copy()
df_dist['data_exibicao'] = pd.to_datetime(df_dist['data_exibicao'])
df_exib['data_exibicao'] = pd.to_datetime(df_exib['data_exibicao'])

map_dias_dist = df_dist.groupby('cpb')['data_exibicao'].nunique()
map_dias_exib = df_dist.groupby('cpb')['data_exibicao'].nunique()

s_dias_dist = map_dias_dist.reindex(df_cluster.index)
s_dias_exib = map_dias_exib.reindex(df_cluster.index)

df_cluster['dias_cartaz'] = s_dias_dist.combine_first(s_dias_exib)

In [83]:
# Adiciona a coluna `renda_total`
map_rendas = df_lanc_comerciais.groupby('cpb')['renda_total'].sum()

s_map_rendas = map_rendas.reindex(df_cluster.index)
df_cluster['renda_total'] = s_map_rendas

In [84]:
df_obras

,titulo_original,cpb,tipo_obra,classificacao_obra,duracao_total_minutos,ano_producao_inicial,segmento_destinacao_inicial,requerente,uf_requerente,municipio_requerente,cat_duracao
0,"""SOLO""",B0901120600000,FICÇÃO,BRASILEIRA INDEPENDENTE CONSTITUINTE DE ESPAÇO...,72.0,2009,INDEFINIDO,SP FILMES DE SÃO PAULO LTDA,SP,SÃO PAULO,LONGA METRAGEM
3,1983.. O ANO AZUL,B0901024500000,DOCUMENTÁRIO,BRASILEIRA CONSTITUINTE DE ESPAÇO QUALIFICADO,89.7,2009,INDEFINIDO,INVÍDEO PRODUÇÕES CINEMATOGRÁFICAS LTDA.,RS,PORTO ALEGRE,LONGA METRAGEM
6,23 ANOS EM 7 SEGUNDOS: 1977 - O FIM DO JEJUM C...,B0901028800000,DOCUMENTÁRIO,BRASILEIRA INDEPENDENTE CONSTITUINTE DE ESPAÇO...,84.0,2009,SALAS DE EXIBIÇÃO,CANAL AZUL CONSULTORIA AUDIOVISUAL - EIRELI,SP,SÃO PAULO,LONGA METRAGEM
7,50 ANOS DEPOIS - AGNALDO RAYOL,B0901003800000,VÍDEOMUSICAL,BRASILEIRA CONSTITUINTE DE ESPAÇO QUALIFICADO,120.0,2009,INDEFINIDO,MAGUS CINEMA E VIDEO LTDA.,SP,SÃO PAULO,LONGA METRAGEM
8,A APARIÇÃO DE MÁRIO,B0901057400000,ANIMAÇÃO,NÃO INFORMADO,4.9,2009,INDEFINIDO,LUANNE CASTRO ARAUJO,RJ,RIO DE JANEIRO,CURTA METRAGEM
...,...,...,...,...,...,...,...,...,...,...,...
33357,É TUDO PARENTE,B2500249300000,DOCUMENTÁRIO,BRASILEIRA INDEPENDENTE CONSTITUINTE DE ESPAÇO...,90.6,2025,SALAS DE EXIBIÇÃO,FABULOSA PRODUCOES LTDA - ME,MG,BELO HORIZONTE,LONGA METRAGEM
33358,ÉRAMOS FELIZES NA CASINHA BRANCA AO PÉ DA SERRA,B2500310100000,DOCUMENTÁRIO,BRASILEIRA INDEPENDENTE CONSTITUINTE DE ESPAÇO...,3.0,2025,OUTROS MERCADOS,50.912.764 FRANCIELITO DA SILVA LIMA,RN,LAGOA NOVA,CURTA METRAGEM
33359,ÊXODO RURAL,B2500143000000,DOCUMENTÁRIO,BRASILEIRA CONSTITUINTE DE ESPAÇO QUALIFICADO,11.8,2025,OUTROS MERCADOS,GABRIELA LACERDA BENELLI,MG,ESPERA FELIZ,CURTA METRAGEM
33360,Ô ABRE ALAS,B2500042000000,VÍDEOMUSICAL,BRASILEIRA INDEPENDENTE CONSTITUINTE DE ESPAÇO...,3.4,2025,VÍDEO POR DEMANDA,AVENIDA LAMPARINA PRODUCOES LTDA,SC,JARAGUÁ DO SUL,CURTA METRAGEM


In [85]:
df_cluster.dropna().count()

investimento_fsa    1741
coproducao          1741
fomento_indireto    1741
publico_total       1741
qtd_sessoes         1741
dias_cartaz         1741
renda_total         1741
curta_metragem      1741
longa_metragem      1741
media_metragem      1741
animacao            1741
documentario        1741
ficcao              1741
variedades          1741
videomusical        1741
dtype: int64

In [86]:
df_cluster.head()

,investimento_fsa,coproducao,fomento_indireto,publico_total,qtd_sessoes,dias_cartaz,renda_total,curta_metragem,longa_metragem,media_metragem,animacao,documentario,ficcao,variedades,videomusical
cpb,,,,,,,,,,,,,,,
B0901120600000,False,False,False,986.0,NaN,NaN,8970.00,False,True,False,False,False,True,False,False
B0901024500000,False,False,False,2313.0,NaN,NaN,23001.98,False,True,False,False,True,False,False,False
B0901028800000,False,False,True,1718.0,NaN,NaN,14936.00,False,True,False,False,True,False,False,False
B0901003800000,False,False,False,NaN,NaN,NaN,NaN,False,True,False,False,False,False,False,True
B0901057400000,False,False,False,NaN,NaN,NaN,NaN,True,False,False,True,False,False,False,False
